In [0]:
df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load("/Volumes/datamodeling/default/practice_file/Practice_file.csv")

display(df)

In [0]:
df_clean = df.toDF(*[
    c.strip()
     .replace(" ", "_")
     .replace(".", "")
     .replace("-", "_")
     .replace("/", "_")
     .replace("(", "")
     .replace(")", "")
     .replace("=", "")
     .replace("'", "")
    for c in df.columns
])

df_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("datamodeling.default.practice_table")

In [0]:
display(spark.read.table("datamodeling.default.practice_table"))

In [0]:
%sql
DESCRIBE DETAIL datamodeling.default.practice_table;

In [0]:
%sql
UPDATE datamodeling.default.practice_table
SET Units_Sold = 50
WHERE Cust_ID = 'C-973342';

In [0]:
%sql
DESCRIBE HISTORY datamodeling.default.practice_table;

In [0]:
%sql
UPDATE datamodeling.default.practice_table
SET Units_Sold = 100
WHERE Cust_ID = 'C-880500';

In [0]:
%sql
DESCRIBE HISTORY datamodeling.default.practice_table;

In [0]:
%sql
SELECT *
FROM datamodeling.default.practice_table VERSION AS OF 1;

In [0]:
# from pyspark.sql.functions import lit

# df_new = df_clean.withColumn("Country", lit("India"))

# df_new.write.format("delta") \
#     .mode("append") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("datamodeling.default.practice_table")

In [0]:
spark.read.table("datamodeling.default.practice_table").printSchema()

**New Incremental data**

In [0]:
cdc_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load("/Volumes/datamodeling/default/practice_file/Practice_file - Incremental.csv")
display(cdc_df)

In [0]:
cdc_df = cdc_df.toDF(*[
    c.strip()
     .replace(" ", "_")
     .replace(".", "")
     .replace("-", "_")
     .replace("/", "_")
     .replace("(", "")
     .replace(")", "")
     .replace("=", "")
     .replace("'", "")
    for c in cdc_df.columns
])

In [0]:
cdc_df.createOrReplaceTempView("cdc_updates")

In [0]:
%sql
MERGE INTO datamodeling.default.practice_table tgt
USING cdc_updates src
ON tgt.Cust_ID = src.Cust_ID

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

In [0]:
df = spark.read.table("datamodeling.default.practice_table")

display(df)